### Next Word Generator

- LLMs are trained on large text corpora.
- Here we use 25 sentences to mimic next-token behavior across real business domains.
- Preprocessing can include removing punctuation, stopwords, and building a vocabulary. No extra preprocessing is needed for this toy corpus.

In [1]:
sentences = [
    # Finance
    "stock prices rise when earnings beat expectations",
    "interest rates affect mortgage and loan affordability",
    "loan default rates rise when unemployment is high",
    "portfolio diversification reduces investment risk",
    "quarterly earnings drive investor decisions and market movement",
    # Marketing
    "customer churn increases when satisfaction scores drop",
    "conversion rates improve with targeted email campaigns",
    "ad spend increases during seasonal peak periods",
    "brand engagement metrics reflect campaign effectiveness",
    "customer segmentation improves marketing campaign results",
    # Healthcare
    "patient readmission rates drop when discharge planning improves",
    "clinical trials measure drug efficacy against control groups",
    "early diagnosis reduces treatment costs significantly",
    "hospital bed occupancy affects patient wait times",
    "preventive care programs lower chronic disease prevalence",
    # Education
    "student engagement improves when course content is personalized",
    "graduation rates increase with early intervention programs",
    "online learning platforms expand access to quality education",
    "teacher feedback improves student performance and outcomes",
    "enrollment declines when tuition costs rise sharply",
    # HR / Retail / Supply Chain
    "employee turnover increases when job satisfaction drops",
    "inventory shortages disrupt retail sales and customer experience",
    "delivery delays increase customer complaint rates sharply",
    "hiring pipelines shorten with automated screening tools",
    "sales forecasting accuracy improves with historical data analysis"
]

Tokenization

In [2]:
tokenized_sentences = [sentence.split() for sentence in sentences]
print(tokenized_sentences[:3])

[['stock', 'prices', 'rise', 'when', 'earnings', 'beat', 'expectations'], ['interest', 'rates', 'affect', 'mortgage', 'and', 'loan', 'affordability'], ['loan', 'default', 'rates', 'rise', 'when', 'unemployment', 'is', 'high']]


This output shows the tokenizer split the first sentences into word-level tokens, which is the input representation used by the simple next-word model.

Vocabulary

In [3]:
words = [word for sentence in tokenized_sentences for word in sentence]
vocab = sorted(set(words))

word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}

print("Vocabulary size:", len(vocab))
print("Sample vocab:", vocab[:10])

Vocabulary size: 134
Sample vocab: ['access', 'accuracy', 'ad', 'affect', 'affects', 'affordability', 'against', 'analysis', 'and', 'automated']


This output confirms the vocabulary was built correctly and shows the token inventory the model can predict from.

Context Window and Target Generation

In [4]:
context_size = 2

contexts = []
targets = []

for sentence in tokenized_sentences:
    for i in range(len(sentence) - context_size):
        context = sentence[i:i+context_size]
        target = sentence[i+context_size]

        contexts.append(context)
        targets.append(target)

Sliding Window

In [5]:
for i in range(10):
    print(f"{contexts[i]} ---> {targets[i]}")

['stock', 'prices'] ---> rise
['prices', 'rise'] ---> when
['rise', 'when'] ---> earnings
['when', 'earnings'] ---> beat
['earnings', 'beat'] ---> expectations
['interest', 'rates'] ---> affect
['rates', 'affect'] ---> mortgage
['affect', 'mortgage'] ---> and
['mortgage', 'and'] ---> loan
['and', 'loan'] ---> affordability


Text to Index

In [6]:
import numpy as np

X = [[word_to_index[word] for word in context] for context in contexts]
y = [word_to_index[word] for word in targets]

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (128, 2)
y shape: (128,)


This output shows the feature matrix and label vector shapes, which means the training data was converted into the expected numeric form.

Index to Embedding vectors

In [7]:
import torch
import torch.nn as nn

class NextWordModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(context_size * embedding_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.shape[0], -1)
        out = self.fc(x)
        return out

Training

In [8]:
vocab_size = len(vocab)
embedding_dim = 10

model = NextWordModel(vocab_size, embedding_dim, context_size)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = loss_fn(outputs, y_tensor)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 5.086235523223877
Epoch 10, Loss: 3.620388984680176
Epoch 20, Loss: 2.3324217796325684
Epoch 30, Loss: 1.2963533401489258
Epoch 40, Loss: 0.6363655924797058
Epoch 50, Loss: 0.2979537546634674
Epoch 60, Loss: 0.15781842172145844
Epoch 70, Loss: 0.10228968411684036
Epoch 80, Loss: 0.07661440968513489
Epoch 90, Loss: 0.06286484003067017


This output shows the loss falling over epochs, so the simple model is learning the next-word pattern from the toy corpus.

Prediction of next word

In [9]:
import random

def predict_next(context_words):
    if any(word not in word_to_index for word in context_words):
        return random.choice(vocab)

    context_idx = torch.tensor([[word_to_index[word] for word in context_words]])
    output = model(context_idx)
    probabilities = torch.softmax(output[0], dim=0)
    predicted_idx = torch.multinomial(probabilities, num_samples=1).item()
    return index_to_word[predicted_idx]

Results

In [10]:
print(predict_next(["stock", "prices"]))           # finance domain
print(predict_next(["patient", "readmission"]))    # healthcare domain
print(predict_next(["student", "engagement"]))     # education domain
print(predict_next(["employee", "turnover"]))      # HR domain
print(predict_next(["hello", "world"]))            # out-of-vocabulary → random fallback

rise
rates
improves
increases
sharply


This output shows the model producing next-word predictions for known contexts and, for the unknown context, falling back to a random vocabulary word from the learned distribution.

### LLM Runtime Concepts (Engineer View)
- LLMs generate output token-by-token. Each step samples from a probability distribution conditioned on the current context.
- Tokens are sub-word chunks. Token counts differ by model and drive both cost and context limits.
- Context window is the maximum total tokens (prompt + tool outputs + model output). Exceed it and the model truncates or rejects.
- Temperature controls randomness: low = more deterministic, high = more varied.
- Non-determinism is normal with sampling; for repeatability use lower temperature or a provider seed when available.

In [11]:
# Approximate token count using whitespace split (not model-accurate)
sample_text = "Analyze Q3 2024 earnings for Apple and flag any revenue guidance changes versus analyst consensus estimates."
approx_tokens = len(sample_text.split())
print("Approx tokens (whitespace split):", approx_tokens)
print("Note: real tokenizers produce different counts — 'Q3' may be 1-2 tokens, numbers split differently")
print("Use the count_tokens API (shown in the lab below) for accurate pre-call budgeting.")

Approx tokens (whitespace split): 16
Note: real tokenizers produce different counts — 'Q3' may be 1-2 tokens, numbers split differently
Use the count_tokens API (shown in the lab below) for accurate pre-call budgeting.


### Model Families (Product/API View)
- GPT (OpenAI): general-purpose chat and tool-use models accessed via API keys and token pricing.
- Claude (Anthropic): chat-first API with long-context options and usage-based billing.
- Gemini (Google): multimodal models with text, image, and tool integrations through Google APIs.
- Model names, context limits, and pricing change frequently; always check provider docs.

### API Overview and Costs
- Shared structure: HTTPS POST with JSON payload that includes `model`, `messages` (or `input`), `max_tokens`, and `temperature`.
- Auth: OpenAI uses `Authorization: Bearer <OPENAI_API_KEY>`; Anthropic uses `x-api-key: <ANTHROPIC_API_KEY>` plus an `anthropic-version` header.
- Responses return generated text plus a `usage` block with input/output token counts.
- Cost is token-based: cost = input_tokens * input_price + output_tokens * output_price.
- Rate limits are enforced as RPM/TPM; handle HTTP 429 with backoff and retries.

#### Request/Response Shapes (Simplified)

OpenAI Chat Completions request:
```json
{
  "model": "gpt-4o-mini",
  "messages": [
    {"role": "user", "content": "Summarize the credit risks in a high-yield bond portfolio."}
  ],
  "max_tokens": 200,
  "temperature": 0.2
}
```

OpenAI response (trimmed):
```json
{
  "id": "chatcmpl-...",
  "choices": [
    {"message": {"role": "assistant", "content": "Key risks include default risk, liquidity..."}}
  ],
  "usage": {"prompt_tokens": 18, "completion_tokens": 42, "total_tokens": 60}
}
```

Anthropic request:
```json
{
  "model": "claude-sonnet-4-6",
  "max_tokens": 200,
  "temperature": 0.2,
  "messages": [
    {"role": "user", "content": "Summarize the credit risks in a high-yield bond portfolio."}
  ]
}
```

Anthropic response (trimmed):
```json
{
  "id": "msg_...",
  "content": [{"type": "text", "text": "Key risks include default risk, liquidity..."}],
  "usage": {"input_tokens": 18, "output_tokens": 42}
}
```

### Lab: First API Call (Anthropic)
- Step 1: Set `ANTHROPIC_API_KEY` as an environment variable (do not hardcode keys).
- Step 2: Install the SDK if needed.
- Step 3: Run the call, inspect the response, and log token usage and cost.

In [ ]:
# If needed, install the SDK in this environment:
# %pip install anthropic

In [23]:
import os
from anthropic import Anthropic

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("Set ANTHROPIC_API_KEY in your environment.")

client = Anthropic(api_key=api_key)

model = "claude-sonnet-4-6"
prompt = "Summarize the key credit risks in a high-yield corporate bond portfolio in 3 bullet points."

# Accurate token count BEFORE the call — avoids surprise context overflows
token_count = client.messages.count_tokens(
    model=model,
    messages=[{"role": "user", "content": prompt}],
)
print(f"Pre-call token count: {token_count.input_tokens} input tokens")

response = client.messages.create(
    model=model,
    max_tokens=300,
    temperature=0.2,
    messages=[{"role": "user", "content": prompt}],
)

text = response.content[0].text
print("\nResponse:")
print(text)

usage = response.usage
print(f"\nActual usage — Input: {usage.input_tokens}, Output: {usage.output_tokens}")

# claude-sonnet-4-6 pricing: $3.00 input / $15.00 output per 1M tokens
INPUT_PRICE_PER_1M = 3.00
OUTPUT_PRICE_PER_1M = 15.00
cost = (usage.input_tokens / 1_000_000) * INPUT_PRICE_PER_1M + \
       (usage.output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
print(f"Estimated cost (USD): ${cost:.6f}")

Pre-call token count: 29 input tokens

Response:
# Key Credit Risks in a High-Yield Corporate Bond Portfolio

• **Default Risk** — High-yield ("junk") issuers carry sub-investment-grade ratings, meaning they have a materially higher probability of missing interest or principal payments, particularly during economic downturns or periods of financial stress when refinancing becomes difficult.

• **Liquidity & Market Risk** — High-yield bonds trade in thinner, less liquid markets than investment-grade debt, making it harder to exit positions at fair value during volatility; spread widening can cause significant mark-to-market losses even without an actual default.

• **Credit Deterioration & Refinancing Risk** — Issuers may face ratings downgrades driven by weakening fundamentals, rising leverage, or sector headwinds; many high-yield companies rely on periodic debt rollovers, and a tightening credit environment can make refinancing costly or impossible, accelerating distress.

Actual usag

### Lab: First API Call (OpenAI)
- Step 1: Set `OPENAI_API_KEY` as an environment variable.
- Step 2: Install the SDK if needed: `pip install openai`
- Step 3: Run the call, inspect the response, and log token usage and cost.
- Note: OpenAI uses `choices[0].message.content` and `usage.prompt_tokens` / `usage.completion_tokens` — different field names from Anthropic.

In [ ]:
import os

# Requires: pip install openai
# Set OPENAI_API_KEY as an environment variable (never hardcode keys)

openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    print("OPENAI_API_KEY not set — skipping. Set it to run this cell.")
else:
    from openai import OpenAI

    openai_client = OpenAI(api_key=openai_api_key)

    openai_model = "gpt-4o-mini"
    openai_prompt = "Summarize the key risks in a high-yield corporate bond portfolio in 3 bullet points."

    openai_response = openai_client.chat.completions.create(
        model=openai_model,
        max_tokens=300,
        temperature=0.2,
        messages=[{"role": "user", "content": openai_prompt}],
    )

    openai_text = openai_response.choices[0].message.content
    print("Response:")
    print(openai_text)

    usage = openai_response.usage
    print(f"\nActual usage — Input: {usage.prompt_tokens}, Output: {usage.completion_tokens}")

    # gpt-4o-mini pricing: $0.15 input / $0.60 output per 1M tokens
    INPUT_PRICE_PER_1M = 0.15
    OUTPUT_PRICE_PER_1M = 0.60
    cost = (usage.prompt_tokens / 1_000_000) * INPUT_PRICE_PER_1M + \
           (usage.completion_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    print(f"Estimated cost (USD): ${cost:.6f}")

#### Rate Limits and Retry Handling
- Providers enforce RPM (requests per minute) and TPM (tokens per minute) quotas.
- Exceeding them returns HTTP 429 — your code must handle this gracefully.
- Pattern: exponential backoff with jitter — each retry waits 2× longer plus a small random offset.
- Finance use case: nightly batch jobs processing thousands of invoices or trade summaries must be rate-limit-aware or they will silently drop records.

In [25]:
import time
import random

def call_with_retry(client, model, messages, max_retries=5, base_delay=1.0):
    """
    Anthropic API call with exponential backoff on HTTP 429 (rate limit).
    Use this for any batch job: invoice processing, patient record summaries,
    student report generation, or bulk marketing content creation.
    """
    for attempt in range(max_retries):
        try:
            return client.messages.create(
                model=model,
                max_tokens=200,
                messages=messages,
            )
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                if attempt == max_retries - 1:
                    raise
                delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
                print(f"Rate limit hit. Retrying in {delay:.1f}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(delay)
            else:
                raise

# Multi-domain batch examples that would hit rate limits at scale
batch_examples = {
    "Finance":    "Summarize risk factors in this loan application: Credit 680, Income $65k, Debt $18k",
    "Healthcare": "Extract diagnosis codes and medications from: 'Patient presents with T2DM, prescribed Metformin 500mg'",
    "Education":  "Generate a personalized study plan for a student scoring 62% in Algebra and 78% in English",
    "Marketing":  "Write a 2-sentence ad copy for a loyalty rewards program targeting millennial customers",
    "HR":         "Score this resume for a data analyst role on communication, SQL, and Python skills",
}

print("Batch jobs per domain — each would hit rate limits at scale:\n")
for domain, task in batch_examples.items():
    print(f"[{domain}] {task[:80]}...")

print("\nUncomment the loop below to process live with retry logic:")
print("# for domain, task in batch_examples.items():")
print("#     resp = call_with_retry(client, 'claude-sonnet-4-6', [{'role':'user','content':task}])")
print("#     print(f'{domain}: {resp.content[0].text[:100]}')")
print("\nRate limit pattern: 1s → 2s → 4s → 8s → 16s (+ jitter to avoid thundering herd)")

Batch jobs per domain — each would hit rate limits at scale:

[Finance] Summarize risk factors in this loan application: Credit 680, Income $65k, Debt $...
[Healthcare] Extract diagnosis codes and medications from: 'Patient presents with T2DM, presc...
[Education] Generate a personalized study plan for a student scoring 62% in Algebra and 78% ...
[Marketing] Write a 2-sentence ad copy for a loyalty rewards program targeting millennial cu...
[HR] Score this resume for a data analyst role on communication, SQL, and Python skil...

Uncomment the loop below to process live with retry logic:
# for domain, task in batch_examples.items():
#     resp = call_with_retry(client, 'claude-sonnet-4-6', [{'role':'user','content':task}])
#     print(f'{domain}: {resp.content[0].text[:100]}')

Rate limit pattern: 1s → 2s → 4s → 8s → 16s (+ jitter to avoid thundering herd)


### LLM Failures

#### Context Window Limits
- Symptoms: earlier instructions are ignored, the model "forgets" prior context, or requests fail with a context-length error.
- Real example: stuffing an entire year of customer support transcripts into a single prompt.
- Engineering fixes: trim prompts, summarize history, use retrieval (RAG), and budget tokens with `count_tokens` before each call.

In [ ]:
# Context window overflow: naive approach vs. engineering fix
# Domain: Healthcare — an EHR system tries to send a patient's full 5-year medical history in one prompt

patient_records = []
for i in range(5000):
    patient_records.append(
        f"Visit {i+1} | Patient ID: P{10000+i} | "
        f"Diagnosis: {'Hypertension' if i % 3 == 0 else 'Type 2 Diabetes' if i % 3 == 1 else 'Routine checkup'} | "
        f"Medication: {'Lisinopril 10mg' if i % 3 == 0 else 'Metformin 500mg' if i % 3 == 1 else 'None'} | "
        f"Notes: Patient reported {'elevated BP readings' if i % 3 == 0 else 'high fasting glucose' if i % 3 == 1 else 'no new symptoms'}."
    )

full_history = "\n".join(patient_records)
approx_tokens = len(full_history.split())

max_context_tokens = 200_000     # claude-sonnet-4-6 context window
reserved_for_output = 2_000
available_for_prompt = max_context_tokens - reserved_for_output

print(f"Full patient record history: ~{approx_tokens:,} tokens")
print(f"Available prompt budget:     {available_for_prompt:,} tokens")

if approx_tokens > available_for_prompt:
    print(f"\nOVERFLOW: exceeds budget by ~{approx_tokens - available_for_prompt:,} tokens")
    print("\nEngineering Fix A — Trim to most recent 10 visits:")
    recent = patient_records[-10:]
    trimmed_tokens = sum(len(r.split()) for r in recent)
    print(f"  Trimmed prompt: ~{trimmed_tokens} tokens — fits within budget")
    print("\nEngineering Fix B — RAG: embed all visits, retrieve the 5 most relevant to the query")
    print("  e.g., query = 'recent medication changes' → fetch only medication-related visits")
    print("\nThis pattern applies to: legal case files, student transcripts, customer support logs, financial statements")
else:
    print("Within context window.")

In [13]:
from langchain_groq.chat_models import ChatGroq
import os

Groq_Token = os.getenv("CHAT_GROQ_API_KEY")

llm = ChatGroq(
    api_key=Groq_Token,
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

Hallucinations

In [14]:
from langchain_core.messages import HumanMessage

prompt = """
You are a clinical research scientist.

Provide detailed results from the Phase 3 clinical trial of NeoCortixel (NCX-7740),
a novel Alzheimer's treatment developed by Meridian BioTherapeutics, published in
The Lancet Neurology in September 2024. Include:
- Trial size (number of participants)
- Primary endpoint result (MMSE score improvement over placebo)
- Adverse event rate (%)
- FDA approval status

Use precise clinical language with specific statistics.
"""

response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)

The Phase 3 clinical trial of NeoCortixel (NCX-7740), a novel Alzheimer's treatment developed by Meridian BioTherapeutics, was published in The Lancet Neurology in September 2024. The trial, titled "Efficacy and Safety of NCX-7740 in Patients with Mild to Moderate Alzheimer's Disease," enrolled a total of 1,044 participants.

The primary endpoint of the study was the change from baseline to week 52 in the Mini-Mental State Examination (MMSE) score, a widely used measure of cognitive function. The results showed that patients treated with NCX-7740 demonstrated a statistically significant improvement in MMSE score compared to those receiving placebo. Specifically, the NCX-7740 group exhibited a mean MMSE score improvement of 3.4 points (95% CI, 2.5-4.3; p < 0.0001) over the placebo group at week 52.

In terms of safety, the adverse event rate was 34.6% in the NCX-7740 group, compared to 26.1% in the placebo group. The most common adverse events reported in the NCX-7740 group were headach

This output is a hallucination: NeoCortixel, NCX-7740, and Meridian BioTherapeutics do not exist. The model generated a trial size, MMSE statistics, adverse event rates, and an FDA status that look clinically credible — exactly the risk when LLMs are used in healthcare decision support or drug research workflows.

The drug, trial, and publisher were all fabricated, yet the model produced structured clinical data with specific statistics. In healthcare, a hallucinated trial result could influence prescribing decisions, mislead researchers, or be misused in regulatory submissions. The same pattern applies to finance (fake earnings), legal (fake case citations), and education (fake accreditation status).

Mitigations (Hallucinations)
- Require citations or source snippets and verify them.
- Use retrieval (RAG) or tools to ground answers in data.
- Add validation and fallback flows for high-risk outputs.

Stale Knowledge

In [15]:
prompt = """
What is the current federal student loan interest rate for undergraduate students in the US?
When did it last change, and what was the previous rate?
"""

response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)

The current federal student loan interest rate for undergraduate students in the US is 4.99% for the 2022-2023 academic year for Direct Subsidized and Unsubsidized Loans. 

The interest rates last changed on July 1, 2022. The previous rate was 3.73% for the 2021-2022 academic year. 

Please note that interest rates may change annually and are determined by the US government. For the most up-to-date information, I recommend checking the official Federal Student Aid website or other reliable sources.


This output shows stale knowledge: US federal student loan rates reset every July 1 based on the 10-year Treasury yield. The model will report whatever rate was current at its training cutoff, which may be one or more academic years behind. An EdTech platform serving this data to prospective students would be giving them incorrect financial guidance.

Stale knowledge affects every domain: an LLM asked about current drug formularies, insurance reimbursement codes, tax brackets, school admission policies, or interest rates will confidently give outdated answers. Always inject live data from authoritative sources (government APIs, provider portals) rather than relying on the model's frozen training snapshot.

Mitigations (Stale Knowledge)
- Use live data sources and retrieval for time-sensitive info.
- Add freshness checks and disclaimers in the UX.
- Log and monitor drift in answers over time.

Non-determinism

In [21]:
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=Groq_Token,
    model="llama-3.3-70b-versatile",
    temperature=1.2
)

prompt = """
A hospital HR team must choose ONE candidate for a single Nurse Practitioner opening.

Candidate A: 7 years ICU experience, BSN + MSN, no specialty certification
Candidate B: 4 years outpatient experience, BSN, certified Family Nurse Practitioner (FNP-BC)
Candidate C: 2 years ER experience, BSN + MSN, certified Adult-Gerontology NP (AGPCNP-BC)

The role involves both inpatient and outpatient care across age groups.
Respond with only the letter of the selected candidate: "A", "B", or "C"
"""

results = []

for i in range(15):
    response = llm.invoke([HumanMessage(content=prompt)])
    results.append(response.content.strip())

print(results)
print("\nUnique:", set(results))

['C', 'B', 'B', 'B', 'B', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'B']

Unique: {'C', 'B'}


This output shows non-determinism: across 15 identical calls, the model returns different hiring recommendations for the same Nurse Practitioner candidates. At temperature=1.2, two HR managers running the same query on the same day could get conflicting shortlists — without any change in inputs.

Non-determinism in high-stakes decisions creates auditability and fairness risks. This applies across domains: inconsistent loan approvals (finance), varying treatment recommendations (healthcare), different student admission outcomes (education), and changing hiring decisions (HR). In regulated industries, the same input must reliably produce the same output — lower temperature or add deterministic post-processing.

Mitigations (Non-determinism)
- Lower `temperature` or set a provider seed if available.
- Use deterministic post-processing or scoring.
- Cache approved outputs for repeated requests.